# 001. Mitigation of Extreme Class Imbalance (in Binary Classifiers)

## Simplified Overview

_When you're trying to predict a rare event (e.g., "Will this vulnerability be exploited?" where exploitation occurs in only 5% of cases), standard machine learning models fail because they learn to always predict "no" and still achieve 95% accuracy._ Techniques like resampling, cost-weighted models, and custom threshold tuning help the model actually learn the rare class.

## Technical Overview

**Class imbalance** occurs when the distribution of binary outcomes is skewed: P(y=1) << P(y=0). In extreme imbalance scenarios (e.g., 1:100 or 1:1000 ratio), standard maximum likelihood estimation produces classiﬁers that:
• Achieve high accuracy through trivial prediction of the majority class
• Fail to learn discriminative patterns for the minority class
• Generate biased probability calibration

Key metrics shift from accuracy to: **ROC-AUC**, **PR-AUC** (Precision-Recall), **F1-score**, and **G-
mean**. _The choice between resampling methods (oversampling, undersampling, SMOTE) and cost-sensitive learning depends on computational constraints and data availability._

Standard classiﬁers optimize a loss function:

$$L\,=\,-\sum\,\left[\,y \cdot log\left(\hat{y}\right)\,+\,\left(1\,-\,y\right) \cdot log\left(1\,-\,\hat{y}\right)\right]$$

With extreme class imbalance, the gradient contribution from minority examples is negligible. The model learns to predict P(y=1) ≈ π (the base rate), effectively ignoring features.

More details on evaluation metrics for imbalanced binary classification tasks:
* ['Tour of Evaluation Metrics for Imbalanced Classification' by Jason Brownlee at _Machine Learning Mastery_](https://machinelearningmastery.com/tour-of-evaluation-metrics-for-imbalanced-classification/)
* ['ROC Curves and Precision-Recall Curves for Imbalanced Classification' by Jason Brownlee at _Machine Learning Mastery_](https://machinelearningmastery.com/roc-curves-and-precision-recall-curves-for-imbalanced-classification/)

## Potential Mitigators for Extreme Class Imbalance

### Resampling Techniques

* **Oversampling**: Duplicate minority examples (risk: overﬁtting)
* **Undersampling**: Remove majority examples (risk: information loss)
* **SMOTE** (Synthetic Minority Over-sampling TEchnique): Interpolate synthetic minority examples along feature space between existing minority points (risks: synthetic noise; not useful for categoricals -- cf. SMOTE-NC for _combination_ of numericals and categoricals)
* **Stratified k-Fold**: Preserve class ratio in train/test splits (risk: computational cost)

### Cost-Sensitive Learning

Modify loss to weight minority misclassiﬁcations higher:

$$L_{weighted}\,=\,-\sum\,\left[\,w_1 \cdot y \cdot log\left(\hat{y}\right)\,+\,w_0 \cdot \left(1\,-\,y\right) \cdot log\left(1\,-\,\hat{y}\right)\right]$$

where w₁ = n₀/n (weight for minority), w₀ = n₁/n (weight for majority)

### Threshold Tuning

Standard classiﬁers output P(y=1) using threshold 0.5. With imbalance, optimal threshold shifts. Use _ROC-AUC_/_PR-AUC_ curve to select threshold maximizing the metric relevant to your business goal.

### Ensemble Methods

* **Balanced Random Forest**: Each tree trained on balanced bootstrap samples
* **Easy Ensemble**: Train multiple classiﬁers on di�erent balanced subsamples
* **Balanced Bagging**: Bootstrap minority class, random sample majority class (notes: bagging typically used to reduce overfitting in high-variance scenarios; listed as an ensemble method due to the use of multiple classifiers in parallel, with resampling)

## Python Libraries

| Library | Purpose |
| :--- | :---: |
| `imbalanced-learn` |  SMOTE, undersampling, balanced ensemble methods |
| `scikit-learn` | class_weight='balanced', stratified splitting |
| `xgboost` / `lightgbm` | Built-in scale_pos_weight for cost-sensitive learning |
| `optuna` | Hyperparameter tuning including threshold optimization |
| `sklearn.metrics` (`roc_curve`, `precision_recall_curve`) | Threshold selection analysis |

## References

* Chawla, N. V., et al. (2002). "SMOTE: Synthetic Minority Over-sampling Technique." _Journal of Artiﬁcial Intelligence Research_, **16**, 321-357.
* He, H., & Garcia, E. A. (2009). "Learning from Imbalanced Data." _IEEE Transactions on Knowledge and Data Engineering_, **21**(9), 1263-1284.
* Fawcett, T. (2006). "An Introduction to ROC Analysis." _Pattern Recognition Letters_, **27**(8), 861-874.

## Python Implementation Examples

### Imporation of Requisite Python Modules

In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline

### Imbalanced Dataset Generation

All source variables numerical

In [2]:
X, y = make_classification(
n_samples=1000,
n_features=20,
n_informative=10,
n_redundant=5,
weights=[0.95, 0.05],
# 95% class 0, 5% class 1
random_state=42
)
print(f"Class distribution: {np.bincount(y)}")

Class distribution: [942  58]


### Test-Train Split with Stratification and Standardized-Scaling

In [4]:
# Split data with stratification to preserve class ratio
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.2, stratify=y, random_state=42
)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Example 1: Cost-weighted Logistic Regression

Weight inversely proportional to class frequency

In [5]:
class_weights = {0: 1, 1: (len(y_train[y_train==0]) / len(y_train[y_train==1]))}
lr_weighted = LogisticRegression(
class_weight='balanced', # Equivalent to manual weighting above
max_iter=1000,
random_state=42
)
lr_weighted.fit(X_train_scaled, y_train)
y_pred_weighted = lr_weighted.predict_proba(X_test_scaled)[:, 1]
print(f"Weighted LR ROC-AUC: {roc_auc_score(y_test, y_pred_weighted):.4f}")

Weighted LR ROC-AUC: 0.8409


### Example 2: SMOTE + Logistic Regression

In [8]:
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)
print(f"After SMOTE: {np.bincount(y_train_smote)}")

# Balanced classes
lr_smote = LogisticRegression(max_iter=1000, random_state=42)
lr_smote.fit(X_train_smote, y_train_smote)
y_pred_smote = lr_smote.predict_proba(X_test_scaled)[:, 1]
print(f"SMOTE + LR ROC-AUC: {roc_auc_score(y_test, y_pred_smote):.4f}")

After SMOTE: [754 754]
SMOTE + LR ROC-AUC: 0.8271


### Example 3: Combined Pipeline with Both Oversampling and Undersampling

In [7]:
pipeline = Pipeline([
('over', SMOTE(sampling_strategy=0.5, random_state=42)),
# Over to 50%
('under', RandomUnderSampler(sampling_strategy=0.8, random_state=42)),
# Under to 80
('clf', LogisticRegression(max_iter=1000, random_state=42))
])
pipeline.fit(X_train_scaled, y_train)
y_pred_pipeline = pipeline.predict_proba(X_test_scaled)[:, 1]
print(f"Pipeline ROC-AUC: {roc_auc_score(y_test, y_pred_pipeline):.4f}")

# Threshold Tuning: Find optimal threshold using Precision-Recall curve
precision, recall, thresholds = precision_recall_curve(y_test, y_pred_pipeline)
pr_auc = auc(recall, precision)
print(f"PR-AUC: {pr_auc:.4f}")

# F1-score optimal threshold
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
optimal_threshold = thresholds[np.argmax(f1_scores[:-1])]
print(f"Optimal threshold (F1): {optimal_threshold:.4f}")

# Apply optimal threshold
y_pred_tuned = (y_pred_pipeline >= optimal_threshold).astype(int)
from sklearn.metrics import f1_score, precision_score, recall_score
print(f"Tuned F1-Score: {f1_score(y_test, y_pred_tuned):.4f}")
print(f"Tuned Precision: {precision_score(y_test, y_pred_tuned):.4f}")
print(f"Tuned Recall: {recall_score(y_test, y_pred_tuned):.4f}")

Pipeline ROC-AUC: 0.8302
PR-AUC: 0.2062
Optimal threshold (F1): 0.5210
Tuned F1-Score: 0.3774
Tuned Precision: 0.2439
Tuned Recall: 0.8333
